In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.metrics import roc_auc_score
import json

# Loading data

In [2]:
df = pd.read_csv("Training data/train.csv")
df_test = pd.read_csv("Training data/test.csv")
df_original = pd.read_csv('Training data/loan_dataset_20000.csv')

target = 'loan_paid_back'

In [3]:
def create_interaction(X, col_1, col_2):
    # astype(str) has to be used to prevent typeerror
    new_interaction = X[col_1].astype(str) + "|" + X[col_2].astype(str)
    return new_interaction

# Preprocessing

Splitting grade and subgrade into two separate features to help model distinguish between them better.

In [4]:
df['subgrade'] = df['grade_subgrade'].str[1]
df['grade'] = df['grade_subgrade'].str[0]

df_test['subgrade'] = df_test['grade_subgrade'].str[1]
df_test['grade'] = df_test['grade_subgrade'].str[0]

Enriching the current training set with historical loan repayment probabilities for key categorical features using Target Encoding on the original dataset. This helps the model establish a stronger baseline for specific categories.

In [5]:
original_cols = ['interest_rate', 'gender', 'loan_purpose', 'education_level', 'grade_subgrade', 'credit_score',
                 'debt_to_income_ratio', 'loan_amount', 'annual_income', 'employment_status', 'marital_status']
orig_cols = []

for col in original_cols:
    map_orig = df_original.groupby(col)['loan_paid_back'].mean().to_dict()

    new_col_name = f'orig_{col}_te'

    df[new_col_name] = df[col].map(map_orig)
    df_test[new_col_name] = df_test[col].map(map_orig)

    local_mean = df[target].mean()
    orig_cols.append(new_col_name)

Features like annual_income and loan_amount exhibit high cardinality and weak linear correlation with the target. Rounding these values to the nearest ones and tens reduces noise and artificially lowers the number of classes, facilitating better split points for tree-based algorithms. This effect is crucial for the subsequent binning process.

In [6]:
# rounding
num_columns = ['annual_income', 'loan_amount']

for col in num_columns:
    for frame in [df, df_test]:
        frame[f"{col}_round_1s"] = frame[col].round(0).astype(int)
        frame[f"{col}_round_10s"] = frame[col].round(-1).astype(int)

In [7]:
num_columns = df.select_dtypes(include=[np.number]).drop(columns=[target, 'id']).columns.tolist()

Applying Frequency Encoding transforms categorical identifiers into density measures. This allows the XGBoost model to directly capture feature distributions, making it easier to isolate rare anomalies from common, safe patterns.

In [8]:
# frequency encoding
freq_columns = []
for col in df.drop(columns=['id', target]).columns:
    freq_map = df[col].value_counts()
    df[f'{col}_freq'] = df[col].map(freq_map)
    df_test[f'{col}_freq'] = df_test[col].map(freq_map).fillna(freq_map.mean())
    freq_columns.append(f"{col}_freq")

    # defragmenting
df = df.copy()
df_test = df_test.copy()

In [9]:
#binning
for col in num_columns:
    for q in [5, 10, 15]:
        res, bins = pd.qcut(df[col], q=q, labels=False, retbins=True, duplicates='drop')
        df[f'{col}_bin{q}'] = res
        df_test[f'{col}_bin{q}'] = pd.cut(df_test[col], bins=bins, labels=False, include_lowest=True)

    # defragmenting
df = df.copy()
df_test = df_test.copy()

Creating new features that have noticeable correlation with target (checked with Spearman rank correlation coefficient).

In [10]:
df['education_level|loan_purpose'] = create_interaction(df, 'education_level', 'loan_purpose')

df_test['education_level|loan_purpose'] = create_interaction(df_test, 'education_level', 'loan_purpose')

In [11]:
df['total_debt'] = df['loan_amount'] + (df['loan_amount'] * (df['interest_rate'] / 100))
df_test['total_debt'] = df_test['loan_amount'] + (df_test['loan_amount'] * (df_test['interest_rate'] / 100))

df['total_debt_income_ratio'] = df['annual_income'] / df['total_debt']
df_test['total_debt_income_ratio'] = df_test['annual_income'] / df_test['total_debt']

Implementing Out-Of-Fold (OOF) Target Encoding for high-cardinality variables. Values are encoded based on a 7-fold cross-validation (optimal number determined empirically). The OOF mechanism is critical here, as it strictly prevents target leakage into the training set, mitigating the risk of severe overfitting.

In [12]:
kf = KFold(n_splits=7, shuffle=True, random_state=42)
y = df[target]
cols = df.drop(columns=['id', target]).columns
for col in cols:
    X_col = df[col]
    mean_encoded = np.zeros(len(df))
    for tr_idx, val_idx in kf.split(df):
        mean_map = y.iloc[tr_idx].groupby(X_col.iloc[tr_idx]).mean()
        mean_encoded[val_idx] = X_col.iloc[val_idx].map(mean_map)

    df[f'mean_{col}'] = mean_encoded

    global_mean_map = y.groupby(X_col).mean()
    df_test[f'mean_{col}'] = df_test[col].map(global_mean_map)

    y_mean = y.mean()
    df[f'mean_{col}'] = df[f'mean_{col}'].fillna(y_mean)
    df_test[f'mean_{col}'] = df_test[f'mean_{col}'].fillna(y_mean)

    # defragmenting (Change to concat later)
df = df.copy()
df_test = df_test.copy()

/tmp/ipykernel_17231/2049445132.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'mean_{col}'] = mean_encoded
/tmp/ipykernel_17231/2049445132.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_test[f'mean_{col}'] = df_test[col].map(global_mean_map)


Converting string columns to categories so XGClassifier can handle them.

In [13]:
df_to_cat = list(df.select_dtypes(include=['str']))

df[df_to_cat] = df[df_to_cat].astype('category')
df_test[df_to_cat] = df_test[df_to_cat].astype('category')

A lot of encoded/binned features weren't that important, so - to increase readability - the essential features were imported to JSON.

In [15]:
with open('model_features.json', 'r') as f:
    selected_features = json.load(f)

X = df[selected_features].drop('id', axis = 1)
y = df[target]
df_test = df_test[selected_features]

# Training

Hyperparameters were optimized using optuna.
A single model with those parameters performed slightly better on the public leaderboard, although I've decided that I'll use OOF ensemble of 15 models as my final model to increase predictions stability.

In [17]:
params = {
    'tree_method': 'hist',
    'device': 'cuda',
    'eval_metric': 'auc',
    'objective': 'binary:logistic',
    'n_estimators': 3500,
    'learning_rate': 0.025114401723736073,
    'max_depth': 3,
    'min_child_weight': 72,
    'subsample': 0.9448853782196552,
    'colsample_bytree': 0.7464156120732282,
    'reg_alpha': 9.857711692301837,
    'reg_lambda': 8.372707644742041e-07,
    'scale_pos_weight': 0.9863690952247113,
    'max_bin': 1816,
}


# model = XGBClassifier(**params, enable_categorical=True)
# model.fit(X, y)

seeds = [42, 69, 420]
cv_scores = []
final_test_preds = np.zeros(len(df_test))

for seed in seeds:
    skf = StratifiedKFold(n_splits= 5, shuffle=True, random_state= seed)

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
        X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

        model = XGBClassifier(**params, random_state=seed, enable_categorical=True)

        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
        print(f"Seed: {seed} \n AUC: {roc_auc_score(y_val, model.predict_proba(X_val)[:,1]):.5f}\n")
        final_test_preds += model.predict_proba(df_test.drop('id', axis = 1))[:, 1] / (len(seeds) * 5)

/home/jakub/kaggle/lib/python3.12/site-packages/xgboost/core.py:751: UserWarning: [18:29:37] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


Seed: 42 
 AUC: 0.92778

Seed: 42 
 AUC: 0.92861

Seed: 42 
 AUC: 0.92630

Seed: 42 
 AUC: 0.92779

Seed: 42 
 AUC: 0.92727

Seed: 69 
 AUC: 0.92712

Seed: 69 
 AUC: 0.92732

Seed: 69 
 AUC: 0.92772

Seed: 69 
 AUC: 0.92882

Seed: 69 
 AUC: 0.92693

Seed: 420 
 AUC: 0.92696

Seed: 420 
 AUC: 0.92869

Seed: 420 
 AUC: 0.92601

Seed: 420 
 AUC: 0.92876

Seed: 420 
 AUC: 0.92746



Saving final predictions into a data frame and exporting them to csv.

In [18]:
final_submission = pd.DataFrame({
    'id': df_test['id'],
    'loan_paid_back': final_test_preds
})
print(f"{final_submission.head()}")

       id  loan_paid_back
0  593994        0.941232
1  593995        0.971240
2  593996        0.510419
3  593997        0.900931
4  593998        0.973911


In [22]:
final_submission.to_csv('Submissions/final_submission.csv', index = False)